# 09a — Deep Feedforward Networks

Corresponds to: Note 09 — Deep Feedforward Network Architectures

**Libraries introduced in this notebook:** No new libraries — numpy only, introduced in Note 03.

**What this notebook demonstrates:**
1. Parameter count formula applied to arbitrary architectures
2. The general L-layer forward pass in numpy
3. Forward pass on the MNIST-sized architecture from Note 09 Part 6 — verifying shapes and output

In [ ]:
import numpy as np

## Part 1: Parameter Count

For a fully connected layer connecting n_in neurons to n_out neurons:

    parameters = n_in × n_out + n_out

We compute this for each layer and sum across the network.
This matches the worked example in Note 09 Part 3.

In [ ]:
def param_count(layer_sizes):
    """layer_sizes: list of ints, e.g. [784, 512, 256, 10]
    Returns total parameter count across all layers."""
    total = 0
    for i in range(1, len(layer_sizes)):
        n_in  = layer_sizes[i - 1]
        n_out = layer_sizes[i]
        layer_params = n_in * n_out + n_out
        print(f"  Layer {i}: {n_in} × {n_out} + {n_out} = {layer_params}")
        total += layer_params
    return total

Architecture A from Note 09 Part 3 — deep and narrow: [4, 8, 8, 8, 2].
Expected total: 202 parameters.

In [ ]:
arch_A = [4, 8, 8, 8, 2]
print("Architecture A — deep narrow:")
total_A = param_count(arch_A)
print(f"  Total: {total_A}")

Architecture B from Note 09 Part 3 — shallow and wide: [4, 64, 2].
Expected total: 450 parameters.

In [ ]:
arch_B = [4, 64, 2]
print("Architecture B — shallow wide:")
total_B = param_count(arch_B)
print(f"  Total: {total_B}")

## Part 2: General L-Layer Forward Pass

The forward pass for any L-layer network follows the same two-step pattern at every layer:

    z⁽ˡ⁾ = W⁽ˡ⁾ᵀ a⁽ˡ⁻¹⁾ + b⁽ˡ⁾
    a⁽ˡ⁾ = f(z⁽ˡ⁾)

We implement this as a loop over layers.
Weights are stored as a list of matrices; biases as a list of vectors.
Variable names match Note 09 notation exactly.

In [ ]:
def relu(z):
    return np.maximum(0, z)

def softmax(z):
    e = np.exp(z - np.max(z))   # subtract max for numerical stability
    return e / e.sum()

def forward_pass(x, weights, biases, hidden_activation=relu, output_activation=softmax):
    """x: input vector (n0,)
    weights: list of W matrices, W[l] has shape (n_{l-1}, n_l)
    biases:  list of b vectors,  b[l] has shape (n_l,)
    Returns: list of (z, a) tuples for each layer, final a is y_hat."""
    cache = []          # stores (z, a) per layer — needed for backprop
    a = x               # a⁽⁰⁾ = x
    L = len(weights)
    for l in range(L):
        W = weights[l]  # shape (n_{l-1}, n_l)
        b = biases[l]   # shape (n_l,)
        z = W.T @ a + b
        a = output_activation(z) if l == L - 1 else hidden_activation(z)
        cache.append((z, a))
        print(f"  Layer {l+1}: z shape={z.shape}, a shape={a.shape}")
    return cache

Test on a small 3-layer network: input size 4, hidden sizes [8, 8], output size 2.
We initialise weights with small random values and verify the shapes are correct.

In [ ]:
rng = np.random.default_rng(42)

layer_sizes = [4, 8, 8, 2]
weights = [rng.normal(0, 0.1, (layer_sizes[i], layer_sizes[i+1]))
           for i in range(len(layer_sizes) - 1)]
biases  = [np.zeros(layer_sizes[i+1])
           for i in range(len(layer_sizes) - 1)]

print("Weight matrix shapes:")
for l, W in enumerate(weights):
    print(f"  W[{l+1}]: {W.shape}")

x = np.array([0.5, -0.3, 0.8, 0.1])
print(f"\nInput x shape: {x.shape}")

print("\nForward pass:")
cache = forward_pass(x, weights, biases)

y_hat = cache[-1][1]
print(f"\nOutput y_hat: {np.round(y_hat, 4)}")
print(f"Sum of y_hat: {y_hat.sum():.6f}  (should be 1.0 — softmax)")

## Part 3: MNIST-Sized Architecture from Note 09 Part 6

The note designs a network for MNIST digit classification:

    Input:    784  (28×28 flattened)
    Hidden 1: 512, ReLU
    Hidden 2: 256, ReLU
    Hidden 3: 128, ReLU
    Output:    10, Softmax

Expected parameter count: 567,434.
We verify the count and run one forward pass to confirm all shapes are correct.

In [ ]:
mnist_sizes = [784, 512, 256, 128, 10]

print("MNIST architecture parameter count:")
total_mnist = param_count(mnist_sizes)
print(f"  Total: {total_mnist}")
print(f"  Expected: 567,434")

Initialise weights and run a forward pass on a single random input (simulating one flattened 28×28 image).

In [ ]:
mnist_weights = [rng.normal(0, 0.01, (mnist_sizes[i], mnist_sizes[i+1]))
                 for i in range(len(mnist_sizes) - 1)]
mnist_biases  = [np.zeros(mnist_sizes[i+1])
                 for i in range(len(mnist_sizes) - 1)]

x_mnist = rng.uniform(0, 1, 784)   # simulated pixel values
print(f"Input shape: {x_mnist.shape}")

print("\nForward pass through MNIST network:")
mnist_cache = forward_pass(x_mnist, mnist_weights, mnist_biases)

y_hat_mnist = mnist_cache[-1][1]
print(f"\nOutput y_hat shape: {y_hat_mnist.shape}")
print(f"Output y_hat (class probabilities): {np.round(y_hat_mnist, 4)}")
print(f"Sum: {y_hat_mnist.sum():.6f}  (should be 1.0)")
print(f"Predicted class: {np.argmax(y_hat_mnist)}")

## Summary

This notebook demonstrated:

1. **Parameter count** — the formula n_{l-1} × n_l + n_l applied layer by layer. Architecture A (deep narrow, [4,8,8,8,2]) uses 202 parameters; Architecture B (shallow wide, [4,64,2]) uses 450. Verified against Note 09 Part 3.
2. **General L-layer forward pass** — the same two-step pattern (z = Wᵀa + b, then a = f(z)) loops over all layers. Weights stored as a list of matrices; cache stores (z, a) per layer for use in backpropagation.
3. **MNIST-sized architecture** — [784, 512, 256, 128, 10] gives 567,434 parameters. One forward pass confirms all shapes are correct and the softmax output sums to 1.